<h1>Summary maker</h1>

In [4]:
#read all data from data_dir
#!/usr/bin/env python3
"""
Stormwatch hourly rainfall compiler (per-station outputs + summary)

What it does
- Reads every CSV in data_dir (each file = one station)
- Uses 'Reading' as timestamp (assumed UTC, tz-naive in file)
- Converts to America/Chicago (DST-aware)
- Aggregates incremental rain to hourly totals (HH:00:00 ... HH:59:59 local)
- Treats NaN Value as 0, but flags the hour with remark "contains NaN"
- Converts inches to mm (25.4)
- Writes:
  (1) One CSV per station: <station>.hourly_mm.csv with columns:
      time_local, rain_mm, remark
  (2) One summary Excel: stormwatch_station_coverage_summary.xlsx
      rows = station, columns = start/end, counts, and missing/present hour ranges (as strings)

Notes
- "Missing" here means: within the GLOBAL min..max hourly timeline across ALL stations,
  this station has no record for that hour (after aggregation).
- If you prefer missing only within each station's own start..end, change MISSING_SCOPE below.
"""

from __future__ import annotations
import re
import sys
from pathlib import Path
import pandas as pd
import numpy as np

In [9]:
#!/usr/bin/env python3
"""
Stormwatch hourly rainfall compiler (per-station outputs + summary)

CHANGES REQUESTED (implemented):
1) Per-station CSV now has BOTH time_local and time_utc columns.
2) Remark now includes NaN count: "contains N NaN"
   - BUT only if hourly_sum_mm > 0 (wet hour). If hourly sum is 0, remark is blank even if NaNs exist.
3) "Missing" in the SUMMARY is now defined by 7-day gaps:
   - A "missing gap" exists only when the time between TWO consecutive PRESENT hours is > 7 days.
   - Shorter gaps are ignored.
   - n_gaps_over_7d, max_gap_days, gap_ranges_over_7d are reported.
4) Summary still includes start/end and counts of present hours.
   - NOTE: n_hours_missing is no longer reported because "missing" is now gap-based.

Output:
- Per station: <station>.hourly_mm.csv with columns:
    time_local, time_utc, rain_mm, remark
- Summary Excel: stormwatch_station_coverage_summary.xlsx with columns:
    station, start_local, end_local, n_hours_present, n_nanflagged_wet_hours,
    n_gaps_over_7d, max_gap_days, gap_ranges_over_7d, present_ranges, per_station_csv
"""

from __future__ import annotations

import sys
from pathlib import Path
import pandas as pd
import numpy as np

# ------------------ USER PATHS ------------------
data_dir = Path("/mnt/12TB/Sujan/Spatial_correlation/Raw_rain_measurement/OneDrive_1_1-26-2026/stormwatch_compiled_rawstyle_mm")
output_dir = Path("/mnt/12TB/Sujan/Spatial_correlation/Codes/Compiled_rain/")
output_dir.mkdir(parents=True, exist_ok=True)

per_station_dir = output_dir / "per_station_hourly"
per_station_dir.mkdir(parents=True, exist_ok=True)

summary_xlsx = output_dir / "stormwatch_station_coverage_summary.xlsx"
SHIFT_HOURS = 11
SHIFT = pd.Timedelta(hours=SHIFT_HOURS)

LOCAL_TZ = "America/Chicago"
IN_TO_MM = 25.4

# Missing definition: only gaps > 7 days between TWO present timestamps
GAP_DAYS = 7
GAP = pd.Timedelta(days=GAP_DAYS)


# ------------------ HELPERS ------------------
def ranges_to_string(ranges: list[tuple[pd.Timestamp, pd.Timestamp]], max_items: int = 50) -> str:
    if not ranges:
        return ""
    parts = []
    for (a, b) in ranges[:max_items]:
        parts.append(f"{a.strftime('%Y-%m-%d %H:%M')} to {b.strftime('%Y-%m-%d %H:%M')}")
    if len(ranges) > max_items:
        parts.append(f"... ({len(ranges)} total ranges)")
    return " | ".join(parts)


def segments_from_times(times: pd.DatetimeIndex, gap: pd.Timedelta) -> list[tuple[pd.Timestamp, pd.Timestamp]]:
    """
    Split into contiguous segments where consecutive present timestamps are <= gap apart.
    Returns list of (start, end) inclusive.
    """
    if len(times) == 0:
        return []
    times = times.sort_values()
    diffs = times[1:] - times[:-1]
    breaks = diffs > gap

    segs = []
    start = times[0]
    for i, is_break in enumerate(breaks, start=1):
        if is_break:
            segs.append((start, times[i - 1]))
            start = times[i]
    segs.append((start, times[-1]))
    return segs


def gaps_over_threshold(times: pd.DatetimeIndex, gap: pd.Timedelta) -> list[tuple[pd.Timestamp, pd.Timestamp, float]]:
    """
    Return list of gaps where consecutive present timestamps are > gap.
    Each entry: (gap_start, gap_end, gap_days)
      - gap_start = previous present timestamp
      - gap_end   = next present timestamp
      - gap_days  = duration in days
    """
    if len(times) < 2:
        return []
    times = times.sort_values()
    diffs = times[1:] - times[:-1]
    out = []
    for prev_t, next_t, d in zip(times[:-1], times[1:], diffs):
        if d > gap:
            out.append((prev_t, next_t, d.total_seconds() / 86400.0))
    return out

def read_station_csv(fp: Path) -> pd.DataFrame:
    df = pd.read_csv(fp)

    needed = {"Reading", "Value", "Unit"}
    missing = needed - set(df.columns)
    if missing:
        raise ValueError(f"{fp.name}: missing columns {sorted(missing)}. Found: {list(df.columns)}")

    s = df["Reading"].astype(str).str.strip()

    # Parse EVERYTHING as UTC tz-aware (works for tz-naive and tz-aware strings)
    t = pd.to_datetime(s, errors="coerce", utc=True)

    # Fallback for the rows pandas couldn't parse
    mask = t.isna()
    if mask.any():
        s2 = s.loc[mask]

        t2 = pd.to_datetime(s2, format="%m/%d/%Y %I:%M:%S %p", errors="coerce", utc=True)
        m2 = t2.isna()
        if m2.any():
            t2.loc[m2] = pd.to_datetime(s2.loc[m2], format="%m/%d/%Y %I:%M %p", errors="coerce", utc=True)

        t.loc[mask] = t2

    # Drop rows where time still didn't parse
    ok = ~t.isna()
    df = df.loc[ok].copy()
    t = t.loc[ok]

    # Convert UTC -> local (DST handled automatically)
    time_local = t.dt.tz_convert(LOCAL_TZ)

    # Values
    v = pd.to_numeric(df["Value"], errors="coerce")
    nan_flag = v.isna()

    unit = df["Unit"].astype(str).str.strip().str.lower()
    is_in = unit.isin(["in", "inch", "inches"])
    is_mm = unit.isin(["mm", "millimeter", "millimeters"])

    rain_mm = pd.Series(np.nan, index=df.index, dtype="float64")
    rain_mm.loc[is_in] = v.loc[is_in] * IN_TO_MM
    rain_mm.loc[is_mm] = v.loc[is_mm]

    unknown = ~(is_in | is_mm)
    if unknown.any():
        sample = unit.loc[unknown].dropna().unique()[:10]
        raise ValueError(f"{fp.name}: unknown Unit values encountered (sample): {sample}")

    return pd.DataFrame(
        {
            "time_local": time_local,
            "rain_mm": rain_mm.astype("float64"),
            "nan_flag": nan_flag.astype(bool),
        }
    )
TZ_RE = re.compile(r"(Z|[+\-]\d{2}:\d{2}|[+\-]\d{4})$")

def main() -> None:
    
    csv_files = sorted([p for p in data_dir.glob("*.csv") if p.is_file()])
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in: {data_dir}")

    summary_rows = []
    
    for fp in csv_files:
        station_id = fp.stem

        raw = read_station_csv(fp)
        hourly = hourly_aggregate(raw)
        hourly = hourly_aggregate(raw)
        
        # ---- FORCE SHIFT to match your expected record ----
        hourly = hourly.copy()
        hourly.index = hourly.index + SHIFT
        # -----------------------------------------------

        # Present hours are those with non-NaN hourly sum
        present_mask = ~hourly["rain_mm"].isna()
        present_times = hourly.index[present_mask]

        # Write per-station CSV with time_local + time_utc columns
        out_csv = per_station_dir / f"{station_id}.hourly_mm.csv"
        out_df = hourly.reset_index()

        if "time_local" not in out_df.columns:
            out_df = out_df.rename(columns={out_df.columns[0]: "time_local"})

        out_df["time_utc"] = out_df["time_local"].dt.tz_convert("UTC")
        out_df = out_df[["time_local", "time_utc", "rain_mm", "remark"]]
        out_df.to_csv(out_csv, index=False)

        # Summary stats
        if len(present_times) == 0:
            summary_rows.append(
                {
                    "station": station_id,
                    "start_local": "",
                    "end_local": "",
                    "n_hours_present": 0,
                    "n_nanflagged_wet_hours": 0,
                    "n_gaps_over_7d": 0,
                    "max_gap_days": "",
                    "gap_ranges_over_7d": "",
                    "present_ranges": "",
                    "per_station_csv": str(out_csv),
                }
            )
            continue

        start_local = present_times.min()
        end_local = present_times.max()

        # Count wet+NaN flagged hours
        n_nanflagged_wet_hours = int((hourly["remark"] != "").sum())

        # Present segments split only by >7-day gaps (for quick visualization)
        present_segs = segments_from_times(present_times, GAP)

        # Gaps >7 days
        gap_list = gaps_over_threshold(present_times, GAP)
        n_gaps = len(gap_list)
        max_gap_days = max((g[2] for g in gap_list), default=np.nan)

        # Serialize gaps as "prev to next (X days)"
        gap_ranges = []
        for a, b, d in gap_list:
            gap_ranges.append(f"{a.strftime('%Y-%m-%d %H:%M')} to {b.strftime('%Y-%m-%d %H:%M')} ({d:.2f} days)")
        gap_ranges_str = " | ".join(gap_ranges[:50]) + (f" ... ({len(gap_ranges)} total gaps)" if len(gap_ranges) > 50 else "")

        summary_rows.append(
            {
                "station": station_id,
                "start_local": start_local.strftime("%Y-%m-%d %H:%M"),
                "end_local": end_local.strftime("%Y-%m-%d %H:%M"),
                "n_hours_present": int(present_mask.sum()),
                "n_nanflagged_wet_hours": n_nanflagged_wet_hours,
                "n_gaps_over_7d": n_gaps,
                "max_gap_days": (f"{max_gap_days:.2f}" if np.isfinite(max_gap_days) else ""),
                "gap_ranges_over_7d": gap_ranges_str,
                "present_ranges": ranges_to_string(present_segs, max_items=50),
                "per_station_csv": str(out_csv),
            }
        )

    summary_df = pd.DataFrame(summary_rows)

    with pd.ExcelWriter(summary_xlsx, engine="openpyxl") as writer:
        summary_df.to_excel(writer, sheet_name="coverage_summary", index=False)

    print(f"Wrote per-station hourly CSVs to: {per_station_dir}")
    print(f"Wrote summary Excel to: {summary_xlsx}")
    print(f"Stations processed: {len(csv_files)}")
    print(f"Missing definition: only gaps > {GAP_DAYS} days between consecutive present hours")


if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        print(f"ERROR: {e}", file=sys.stderr)
        sys.exit(1)


Wrote per-station hourly CSVs to: /mnt/12TB/Sujan/Spatial_correlation/Codes/Compiled_rain/per_station_hourly
Wrote summary Excel to: /mnt/12TB/Sujan/Spatial_correlation/Codes/Compiled_rain/stormwatch_station_coverage_summary.xlsx
Stations processed: 162
Missing definition: only gaps > 7 days between consecutive present hours


<h1>Plotter</h1>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import re
import math

summary_xlsx = "/mnt/12TB/Sujan/Spatial_correlation/Codes/Compiled_rain/stormwatch_station_coverage_summary.xlsx"
LOCAL_TZ = "America/Chicago"
STATIONS_PER_FIG = 30

# ---------------- LOAD SUMMARY ----------------
df = pd.read_excel(summary_xlsx, sheet_name="coverage_summary")

df["start_dt"] = pd.to_datetime(df["start_local"], errors="coerce")
df["end_dt"]   = pd.to_datetime(df["end_local"], errors="coerce")
df = df.dropna(subset=["start_dt", "end_dt"]).copy()

df["start_dt"] = df["start_dt"].dt.tz_localize(LOCAL_TZ)
df["end_dt"]   = df["end_dt"].dt.tz_localize(LOCAL_TZ)

# Sort stations numerically
df["station_num"] = pd.to_numeric(df["station"], errors="coerce")
df = df.sort_values("station_num").reset_index(drop=True)
df = df.drop(columns="station_num")

# ---------------- PARSE PRESENT RANGES ----------------
range_pat = re.compile(r"(\d{4}-\d{2}-\d{2} \d{2}:\d{2}) to (\d{4}-\d{2}-\d{2} \d{2}:\d{2})")

def parse_ranges(s):
    if not isinstance(s, str) or s.strip() == "":
        return []
    out = []
    for a, b in range_pat.findall(s):
        ta = pd.to_datetime(a).tz_localize(LOCAL_TZ)
        tb = pd.to_datetime(b).tz_localize(LOCAL_TZ)
        out.append((ta, tb))
    return out

df["present_segments"] = df["present_ranges"].apply(parse_ranges)

# ---------------- PLOTTING (BATCHED) ----------------
n_stations = len(df)
n_figs = math.ceil(n_stations / STATIONS_PER_FIG)

for fig_idx in range(n_figs):
    start_i = fig_idx * STATIONS_PER_FIG
    end_i = min((fig_idx + 1) * STATIONS_PER_FIG, n_stations)
    sub = df.iloc[start_i:end_i]

    fig, ax = plt.subplots(
        figsize=(16, max(6, 0.35 * len(sub)))
    )

    for y, row in enumerate(sub.itertuples(index=False)):
        segs = row.present_segments or [(row.start_dt, row.end_dt)]

        for s, e in segs:
            x0 = mdates.date2num(s)
            x1 = mdates.date2num(e)
            if x1 > x0:
                ax.broken_barh(
                    [(x0, x1 - x0)],
                    (y - 0.4, 0.8),
                    facecolors="tab:blue"
                )

    # Y axis
    ax.set_yticks(range(len(sub)))
    ax.set_yticklabels(sub["station"].astype(str).tolist())
    ax.set_ylim(-0.5, len(sub) - 0.5)
    ax.margins(y=0)

    # X axis formatting
    ax.xaxis.set_major_locator(mdates.YearLocator(1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    ax.xaxis.set_minor_locator(mdates.MonthLocator(1))
    ax.xaxis.set_minor_formatter(mticker.NullFormatter())

    ax.grid(which="major", axis="x", linestyle=":", linewidth=0.8, alpha=0.35)

    ax.set_xlabel("Date (America/Chicago)")
    ax.set_ylabel("Station")
    ax.set_title(
        f"Station Data Availability (Stations {start_i+1}–{end_i} of {n_stations})"
    )

    plt.tight_layout()
    plt.show()

<h1>Raw data compiler from three folders</h1>

In [10]:
#!/usr/bin/env python3
from __future__ import annotations

import sys
from pathlib import Path
import pandas as pd
import numpy as np

# ------------------ USER PATHS (EDIT) ------------------
BASE = Path("/mnt/12TB/Sujan/Spatial_correlation/Raw_rain_measurement/OneDrive_1_1-26-2026")

RAW_DIR    = BASE / "stormwatch_raw_csv"
HOURLY_DIR = BASE / "stormwatch_hourly_csv"
EVENT_DIR  = BASE / "stormwatch_event_specific"

OUT_DIR = BASE / "stormwatch_combined"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MISMATCH_REPORT_CSV = OUT_DIR / "mismatch_report.csv"
INVENTORY_CSV       = OUT_DIR / "station_inventory.csv"

# ------------------ OPTIONS ------------------
# Raw/Event files are inches in many cases. Turn ON to compare against hourly mm.
NORMALIZE_RAW_EVENT_TO_MM = True
IN_TO_MM = 25.4

# Strict mismatch check (no tolerance). If you want float tolerance later, set e.g. 1e-9 or 0.001
MISMATCH_TOL = 0.0


# ------------------ HELPERS ------------------
def parse_stormwatch_reading(series: pd.Series) -> pd.Series:
    """
    Fast parser for the confirmed Stormwatch format:
      12/30/2023 11:12:32 AM
    Returns tz-naive timestamps (then we localize to UTC elsewhere).
    """
    s = series.astype(str).str.strip()

    # Primary (your confirmed format)
    t = pd.to_datetime(s, format="%m/%d/%Y %I:%M:%S %p", errors="coerce")

    # Fallback (in case a few rows lack seconds)
    m = t.isna()
    if m.any():
        t.loc[m] = pd.to_datetime(s.loc[m], format="%m/%d/%Y %I:%M %p", errors="coerce")

    # Last resort fallback
    m = t.isna()
    if m.any():
        t.loc[m] = pd.to_datetime(s.loc[m], errors="coerce")

    return t



def list_station_files(folder: Path) -> dict[str, Path]:
    if not folder.exists():
        return {}
    return {p.stem: p for p in folder.glob("*.csv") if p.is_file()}


def standardize_raw_or_event(fp: Path, source: str) -> pd.DataFrame:
    df = pd.read_csv(fp)

    needed = {"Reading", "Value", "Unit"}
    missing = needed - set(df.columns)
    if missing:
        raise ValueError(f"{fp.name} ({source}): missing columns {sorted(missing)}. Found: {list(df.columns)}")

    time = parse_stormwatch_reading(df["Reading"])
    out = pd.DataFrame({"time": time})

    out[f"{source}_raw_value_str"] = df["Value"].astype(str)
    out[f"{source}_value"] = pd.to_numeric(df["Value"], errors="coerce")

    unit = df["Unit"].astype(str).str.strip()
    out[f"{source}_unit"] = unit

    out = out.dropna(subset=["time"])

    # ✅ Make timestamps comparable with hourly (which is parsed as UTC)
    out["time"] = out["time"].dt.tz_localize("UTC")

    if NORMALIZE_RAW_EVENT_TO_MM:
        u = out[f"{source}_unit"].astype(str).str.lower()
        is_in = u.isin(["in", "inch", "inches"])
        is_mm = u.isin(["mm", "millimeter", "millimeters"])

        out.loc[is_in, f"{source}_value"] = out.loc[is_in, f"{source}_value"] * IN_TO_MM
        out.loc[is_in | is_mm, f"{source}_unit"] = "mm"

    return out



def standardize_hourly(fp: Path) -> pd.DataFrame:
    """
    Hourly schema is not confirmed yet; keep safe inference.
    Output columns:
      time, hourly_value
    """
    df = pd.read_csv(fp)

    # time
    if "time_local" in df.columns:
        tcol = "time_local"
    elif "time" in df.columns:
        tcol = "time"
    elif "index" in df.columns:
        tcol = "index"
    else:
        tcol = df.columns[0]

    # value
    if "rain_mm" in df.columns:
        vcol = "rain_mm"
    elif "hourly_value" in df.columns:
        vcol = "hourly_value"
    else:
        # fallback
        if len(df.columns) < 2:
            raise ValueError(f"{fp.name} (hourly): cannot infer value column. Columns: {list(df.columns)}")
        vcol = df.columns[1]

    out = df[[tcol, vcol]].copy()
    out = out.rename(columns={tcol: "time", vcol: "hourly_value"})

    # parse time robustly (handles -06:00 / -05:00 etc)
    out["time"] = pd.to_datetime(out["time"], errors="coerce", utc=True)
    out = out.dropna(subset=["time"])

    out["hourly_value"] = pd.to_numeric(out["hourly_value"], errors="coerce")
    return out


def values_disagree(vals: pd.Series, tol: float) -> bool:
    vmin = vals.min()
    vmax = vals.max()
    return (vmax - vmin) > tol


# ------------------ MAIN ------------------
def main() -> None:
    raw_files = list_station_files(RAW_DIR)
    hourly_files = list_station_files(HOURLY_DIR)
    event_files = list_station_files(EVENT_DIR)

    all_stations = sorted(set(raw_files) | set(hourly_files) | set(event_files))
    if not all_stations:
        raise FileNotFoundError("No CSV station files found across the three folders.")

    # inventory
    inv = pd.DataFrame(
        {
            "station": all_stations,
            "in_raw": [s in raw_files for s in all_stations],
            "in_hourly": [s in hourly_files for s in all_stations],
            "in_event": [s in event_files for s in all_stations],
        }
    )
    inv.to_csv(INVENTORY_CSV, index=False)

    print(f"Stations (union): {len(all_stations)}")
    print(f"raw:   {len(raw_files)}")
    print(f"hourly:{len(hourly_files)}")
    print(f"event: {len(event_files)}")
    print(f"Inventory written: {INVENTORY_CSV}")

    mismatch_rows = []

    for sid in all_stations:
        dfs = []

        if sid in raw_files:
            dfs.append(standardize_raw_or_event(raw_files[sid], "raw"))

        if sid in event_files:
            dfs.append(standardize_raw_or_event(event_files[sid], "event"))

        if sid in hourly_files:
            dfs.append(standardize_hourly(hourly_files[sid]))

        if not dfs:
            continue

        merged = dfs[0]
        for nxt in dfs[1:]:
            merged = merged.merge(nxt, on="time", how="outer")

        merged = merged.sort_values("time").reset_index(drop=True)

        # mismatch check
        value_cols = []
        for c in ["raw_value", "event_value", "hourly_value"]:
            if c in merged.columns:
                value_cols.append(c)

        if len(value_cols) >= 2:
            vals = merged[value_cols]
            candidate = vals.notna().sum(axis=1) >= 2
            cand = merged.loc[candidate, ["time"] + value_cols].copy()

            for _, r in cand.iterrows():
                row_vals = r[value_cols].dropna()
                if values_disagree(row_vals, MISMATCH_TOL):
                    mismatch_rows.append(
                        {
                            "station": sid,
                            "time": r["time"],
                            "n_sources": int(row_vals.shape[0]),
                            "min_value": float(row_vals.min()),
                            "max_value": float(row_vals.max()),
                            "abs_diff": float(row_vals.max() - row_vals.min()),
                            "raw_value": r.get("raw_value", np.nan),
                            "event_value": r.get("event_value", np.nan),
                            "hourly_value": r.get("hourly_value", np.nan),
                        }
                    )

        # write combined file
        out_fp = OUT_DIR / f"{sid}.csv"
        merged.to_csv(out_fp, index=False)

    mismatch_df = pd.DataFrame(mismatch_rows)
    mismatch_df.to_csv(MISMATCH_REPORT_CSV, index=False)

    print(f"Combined files written: {OUT_DIR}")
    print(f"Mismatch report: {MISMATCH_REPORT_CSV}")
    print(f"Mismatch rows: {len(mismatch_df)}")
    print(f"Unit normalization raw/event -> mm: {NORMALIZE_RAW_EVENT_TO_MM}")
    print(f"Mismatch tolerance: {MISMATCH_TOL}")


if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        print(f"ERROR: {e}", file=sys.stderr)
        sys.exit(1)


IndentationError: expected an indented block after 'if' statement on line 145 (2676146678.py, line 146)

<h1>Create files same as raw files but combining all folders</h1>

In [ ]:
#!/usr/bin/env python3
"""
Compile Stormwatch station CSVs from multiple folders into ONE raw-style CSV per station.

User requirements implemented
- Inputs: three folders containing per-station CSVs
- Output: one CSV per station in a new folder, with RAW-style header names:
    Reading, Receive, Value, Unit, Data Quality
  but:
    - Receive is left blank (ignored)
    - Data Quality is left blank (ignored)
- Output values in mm (Unit="mm")
- Timestamp format: ISO (UTC), in Reading column
- No priority between folders: if multiple sources provide a value at the same timestamp:
    - take the average (mean) across available numeric values
    - record conflicts in a report (where sources differ)

Outputs
- compiled_rawstyle_mm/<station>.csv
- compiled_rawstyle_mm/conflict_report.csv
- compiled_rawstyle_mm/station_inventory.csv

Notes
- Raw/Event inputs expected schema: Reading, Receive, Value, Unit, Data Quality
- Hourly inputs expected schema: time_local (or index/time) + rain_mm (or value column)
- All timestamps are normalized to UTC and written as ISO strings (e.g. 2023-12-30T11:12:32Z)
"""

from __future__ import annotations

import sys
from pathlib import Path
import pandas as pd
import numpy as np

# ------------------ USER PATHS ------------------
BASE = Path("/mnt/12TB/Sujan/Spatial_correlation/Raw_rain_measurement/OneDrive_1_1-26-2026")

RAW_DIR    = BASE / "stormwatch_raw_csv"
HOURLY_DIR = BASE / "stormwatch_hourly_csv"
EVENT_DIR  = BASE / "stormwatch_event_specific"

OUT_DIR = BASE / "stormwatch_compiled_rawstyle_mm"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CONFLICT_REPORT = OUT_DIR / "conflict_report.csv"
INVENTORY_CSV   = OUT_DIR / "station_inventory.csv"

# ------------------ CONSTANTS ------------------
IN_TO_MM = 25.4
# conflict tolerance in mm; 0.0 means any difference is a conflict
CONFLICT_TOL_MM = 0.0

# ------------------ PARSERS ------------------
def parse_stormwatch_reading(series: pd.Series) -> pd.Series:
    """
    Confirmed raw/event Reading format: 12/30/2023 11:12:32 AM
    Returns tz-naive timestamps; we localize to UTC afterward.
    """
    s = series.astype(str).str.strip()

    t = pd.to_datetime(s, format="%m/%d/%Y %I:%M:%S %p", errors="coerce")

    m = t.isna()
    if m.any():
        # fallback: no seconds
        t.loc[m] = pd.to_datetime(s.loc[m], format="%m/%d/%Y %I:%M %p", errors="coerce")

    m = t.isna()
    if m.any():
        # last resort
        t.loc[m] = pd.to_datetime(s.loc[m], errors="coerce")

    return t


def list_station_files(folder: Path) -> dict[str, Path]:
    if not folder.exists():
        return {}
    return {p.stem: p for p in folder.glob("*.csv") if p.is_file()}


def load_raw_or_event(fp: Path, source: str) -> pd.DataFrame:
    """
    Load raw/event files and return standardized:
      time_utc (tz-aware), <source>_mm (float)
    """
    df = pd.read_csv(fp)

    needed = {"Reading", "Value", "Unit"}
    missing = needed - set(df.columns)
    if missing:
        raise ValueError(f"{fp.name} ({source}): missing columns {sorted(missing)}. Found: {list(df.columns)}")

    t = parse_stormwatch_reading(df["Reading"])
    v = pd.to_numeric(df["Value"], errors="coerce")

    unit = df["Unit"].astype(str).str.strip().str.lower()

    # Drop bad times
    keep = ~t.isna()
    t = t.loc[keep]
    v = v.loc[keep]
    unit = unit.loc[keep]

    # Localize to UTC (your workflow assumes Reading is UTC)
    t_utc = t.dt.tz_localize("UTC")

    # Convert to mm
    is_in = unit.isin(["in", "inch", "inches"])
    is_mm = unit.isin(["mm", "millimeter", "millimeters"])

    v_mm = pd.Series(np.nan, index=v.index, dtype="float64")
    v_mm.loc[is_in] = v.loc[is_in] * IN_TO_MM
    v_mm.loc[is_mm] = v.loc[is_mm]

    # Unknown units: keep as NaN (and they won't participate in averaging)
    # If you prefer hard-fail, change this to raise.
    # unknown = ~(is_in | is_mm)
    # if unknown.any(): raise ValueError(...)

    out = pd.DataFrame({"time_utc": t_utc, f"{source}_mm": v_mm.astype("float64")})
    out = out.dropna(subset=["time_utc"])
    return out


def load_hourly(fp: Path) -> pd.DataFrame:
    """
    Load hourly files and return standardized:
      time_utc (tz-aware), hourly_mm (float)
    Assumes hourly is already in mm when column is rain_mm.
    """
    df = pd.read_csv(fp)

    # time column
    if "time_utc" in df.columns:
        tcol = "time_utc"
    elif "time_local" in df.columns:
        tcol = "time_local"
    elif "time" in df.columns:
        tcol = "time"
    elif "index" in df.columns:
        tcol = "index"
    else:
        tcol = df.columns[0]

    # value column
    if "rain_mm" in df.columns:
        vcol = "rain_mm"
    elif "hourly_value" in df.columns:
        vcol = "hourly_value"
    else:
        if len(df.columns) < 2:
            raise ValueError(f"{fp.name} (hourly): cannot infer value column. Columns: {list(df.columns)}")
        vcol = df.columns[1]

    out = df[[tcol, vcol]].copy()
    out = out.rename(columns={tcol: "time_utc", vcol: "hourly_mm"})

    # Parse time robustly to UTC (handles mixed offsets)
    out["time_utc"] = pd.to_datetime(out["time_utc"], errors="coerce", utc=True)
    out = out.dropna(subset=["time_utc"])

    out["hourly_mm"] = pd.to_numeric(out["hourly_mm"], errors="coerce").astype("float64")
    return out


def iso_z(ts: pd.Timestamp) -> str:
    """UTC ISO format with Z."""
    # ensure UTC
    if ts.tzinfo is None:
        ts = ts.tz_localize("UTC")
    else:
        ts = ts.tz_convert("UTC")
    return ts.strftime("%Y-%m-%dT%H:%M:%SZ")


def main() -> None:
    raw_files = list_station_files(RAW_DIR)
    hourly_files = list_station_files(HOURLY_DIR)
    event_files = list_station_files(EVENT_DIR)

    stations = sorted(set(raw_files) | set(hourly_files) | set(event_files))
    if not stations:
        raise FileNotFoundError("No station CSVs found in the three input folders.")

    # Inventory
    inv = pd.DataFrame(
        {
            "station": stations,
            "in_raw": [s in raw_files for s in stations],
            "in_hourly": [s in hourly_files for s in stations],
            "in_event": [s in event_files for s in stations],
        }
    )
    inv.to_csv(INVENTORY_CSV, index=False)

    conflict_rows = []

    for sid in stations:
        dfs = []

        if sid in raw_files:
            dfs.append(load_raw_or_event(raw_files[sid], "raw"))
        if sid in event_files:
            dfs.append(load_raw_or_event(event_files[sid], "event"))
        if sid in hourly_files:
            dfs.append(load_hourly(hourly_files[sid]))

        if not dfs:
            continue

        merged = dfs[0]
        for nxt in dfs[1:]:
            merged = merged.merge(nxt, on="time_utc", how="outer")

        merged = merged.sort_values("time_utc").reset_index(drop=True)

        # Columns containing mm values
        val_cols = [c for c in merged.columns if c.endswith("_mm")]  # raw_mm, event_mm, hourly_mm
        if not val_cols:
            continue

        # Compute compiled value (mean across available numeric values)
        merged["Value_mm"] = merged[val_cols].mean(axis=1, skipna=True)

        # Conflict detection: if >=2 sources present and their spread > tolerance
        present_counts = merged[val_cols].notna().sum(axis=1)
        candidates = present_counts >= 2

        if candidates.any():
            cand = merged.loc[candidates, ["time_utc"] + val_cols].copy()
            row_min = cand[val_cols].min(axis=1, skipna=True)
            row_max = cand[val_cols].max(axis=1, skipna=True)
            spread = row_max - row_min

            conflict_mask = spread > CONFLICT_TOL_MM
            if conflict_mask.any():
                sub = cand.loc[conflict_mask].copy()
                sub["station"] = sid
                sub["n_sources"] = present_counts.loc[candidates].loc[conflict_mask].astype(int).values
                sub["min_mm"] = row_min.loc[conflict_mask].values
                sub["max_mm"] = row_max.loc[conflict_mask].values
                sub["spread_mm"] = spread.loc[conflict_mask].values
                conflict_rows.append(sub)

        # Build RAW-style output: Reading, Receive, Value, Unit, Data Quality
        out = pd.DataFrame(
            {
                "Reading": merged["time_utc"].apply(iso_z),
                "Receive": "",          # ignored by your requirement
                "Value": merged["Value_mm"].astype("float64"),
                "Unit": "mm",
                "Data Quality": "",     # ignored by your requirement
            }
        )

        out_fp = OUT_DIR / f"{sid}.csv"
        out.to_csv(out_fp, index=False)

    # Write conflict report
    if conflict_rows:
        conflict_df = pd.concat(conflict_rows, ignore_index=True)

        # Put time in ISO Z too
        conflict_df["time_utc"] = pd.to_datetime(conflict_df["time_utc"], utc=True)
        conflict_df["time_iso"] = conflict_df["time_utc"].apply(iso_z)

        # order columns nicely
        ordered = ["station", "time_iso", "n_sources", "min_mm", "max_mm", "spread_mm"]
        ordered += [c for c in ["raw_mm", "event_mm", "hourly_mm"] if c in conflict_df.columns]

        conflict_df = conflict_df[ordered].sort_values(["station", "time_iso"])
        conflict_df.to_csv(CONFLICT_REPORT, index=False)
    else:
        # empty file with header
        conflict_df = pd.DataFrame(columns=["station", "time_iso", "n_sources", "min_mm", "max_mm", "spread_mm", "raw_mm", "event_mm", "hourly_mm"])
        conflict_df.to_csv(CONFLICT_REPORT, index=False)

    print(f"Compiled raw-style CSVs written to: {OUT_DIR}")
    print(f"Station inventory: {INVENTORY_CSV}")
    print(f"Conflict report: {CONFLICT_REPORT}")
    print(f"Stations compiled: {len(stations)}")
    print(f"Conflict tolerance (mm): {CONFLICT_TOL_MM}")
    if conflict_rows:
        print(f"Conflict rows: {len(conflict_df)}")
    else:
        print("Conflict rows: 0")


if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        print(f"ERROR: {e}", file=sys.stderr)
        sys.exit(1)


<h1>Create hourly bins</h1>

In [11]:
#!/usr/bin/env python3
"""
Aggregate Stormwatch "compiled_rawstyle_mm" station files to hourly totals (mm).

Input folder contains per-station CSVs with RAW-style headers:
  Reading, Receive, Value, Unit, Data Quality
…but in your compiled exports:
  - Reading is ISO UTC like: 2012-01-01T21:00:22Z
  - Value is in mm
  - Unit is "mm"

This script:
- Parses Reading as UTC
- Converts to America/Chicago local time (DST-aware) for hourly binning
- Aggregates to hourly totals (sum of increments in each hour)
- Writes one hourly CSV per station
- Writes a summary CSV of coverage and basic stats

Defaults (edit below if needed):
- Hourly bins are LOCAL time: America/Chicago
- Hour with no records -> rain_mm = NaN (kept in output)
- A remark is added if the hour had any non-numeric Value rows (rare in compiled)
"""

from __future__ import annotations

import sys
from pathlib import Path
import pandas as pd
import numpy as np


# ------------------ PATHS (EDIT IF NEEDED) ------------------
INPUT_DIR = Path("/mnt/12TB/Sujan/Spatial_correlation/Raw_rain_measurement/OneDrive_1_1-26-2026/stormwatch_compiled_rawstyle_mm/")

OUTPUT_DIR = Path("/mnt/12TB/Sujan/Spatial_correlation/Codes/Compiled_rain/compiled_rawstyle_hourly_mm/")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PER_STATION_DIR = OUTPUT_DIR / "per_station_hourly"
PER_STATION_DIR.mkdir(parents=True, exist_ok=True)
SHIFT_HOURS = 11
SHIFT = pd.Timedelta(hours=SHIFT_HOURS)
SUMMARY_CSV = OUTPUT_DIR / "hourly_coverage_summary.csv"

# ------------------ SETTINGS ------------------
LOCAL_TZ = "America/Chicago"     # DST-aware local time
FREQ = "h"                      # hourly
KEEP_FULL_HOURLY_INDEX = True   # include all hours between start and end
MISSING_AS_NAN = True           # hours with no records -> NaN; if False -> 0.0

# If you want to flag NaN values in Value column:
FLAG_NAN_VALUES = True


# ------------------ HELPERS ------------------
def parse_reading_utc(series: pd.Series) -> pd.Series:
    """
    Parse Reading as UTC. Supports:
    - ISO with Z: 2012-01-01T21:00:22Z
    - (fallback) other parsable formats
    Returns tz-aware UTC timestamps; invalid parses become NaT.
    """
    s = series.astype(str).str.strip()
    t = pd.to_datetime(s, errors="coerce", utc=True)
    return t


def read_compiled_station(fp: Path) -> pd.DataFrame:
    df = pd.read_csv(fp)

    needed = {"Reading", "Value", "Unit"}
    missing = needed - set(df.columns)
    if missing:
        raise ValueError(f"{fp.name}: missing columns {sorted(missing)}. Found: {list(df.columns)}")

    t_utc = parse_reading_utc(df["Reading"])
    v = pd.to_numeric(df["Value"], errors="coerce")
    nan_flag = v.isna()

    # Unit handling (expect mm)
    unit = df["Unit"].astype(str).str.strip().str.lower()
    # Accept mm, but don't crash if some blanks exist; just keep values as-is.
    # If you want strict enforcement, uncomment below.
    # if not unit.dropna().isin(["mm", "millimeter", "millimeters"]).all():
    #     bad = unit[~unit.isin(["mm", "millimeter", "millimeters"])].unique()[:10]
    #     raise ValueError(f"{fp.name}: unexpected units (sample): {bad}")

    out = pd.DataFrame(
        {
            "time_utc": t_utc,
            "value_mm": v.astype("float64"),
            "nan_flag": nan_flag.astype(bool),
        }
    ).dropna(subset=["time_utc"])

    # Convert to local time for binning
    out["time_local"] = out["time_utc"].dt.tz_convert(LOCAL_TZ)
    return out[["time_local", "time_utc", "value_mm", "nan_flag"]]


def hourly_aggregate(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate to hourly totals in LOCAL time.
    - rain_mm: sum of value_mm in each hour
    - n_records: number of records in that hour
    - n_nan: number of non-numeric Value rows (if any)
    - remark: optional
    """
    df = df.sort_values("time_local").set_index("time_local")

    # Sum of increments; keep empty hours as NaN if min_count=1
    rain = df["value_mm"].resample(FREQ).sum(min_count=1)

    n_records = df["value_mm"].resample(FREQ).count()  # counts non-NaN numeric values
    n_total = df["value_mm"].resample(FREQ).size()     # total rows in hour (incl NaN)
    n_nan = (n_total - n_records).astype(int)

    # If you want to keep a full continuous hourly index (from min hour to max hour)
    if KEEP_FULL_HOURLY_INDEX and len(rain.index) > 0:
        full_index = pd.date_range(rain.index.min(), rain.index.max(), freq=FREQ, tz=rain.index.tz)
        rain = rain.reindex(full_index)
        n_records = n_records.reindex(full_index).fillna(0).astype(int)
        n_nan = n_nan.reindex(full_index).fillna(0).astype(int)

    if not MISSING_AS_NAN:
        rain = rain.fillna(0.0)

    remark = pd.Series("", index=rain.index, dtype="object")
    if FLAG_NAN_VALUES:
        has_nan = (n_nan > 0)
        remark.loc[has_nan] = n_nan.loc[has_nan].apply(lambda k: f"contains {k} NaN Value")

    out = pd.DataFrame(
        {
            "rain_mm": rain.astype("float64"),
            "n_records": n_records.astype(int),
            "remark": remark.astype(str),
        }
    )
    out.index.name = "time_local"
    return out


def to_iso_z(ts: pd.Timestamp) -> str:
    if ts.tzinfo is None:
        ts = ts.tz_localize("UTC")
    else:
        ts = ts.tz_convert("UTC")
    return ts.strftime("%Y-%m-%dT%H:%M:%SZ")


def main() -> None:
    files = sorted([p for p in INPUT_DIR.glob("*.csv") if p.is_file()])
    if not files:
        raise FileNotFoundError(f"No CSV files found in: {INPUT_DIR}")

    summary_rows = []

    for fp in files:
        station = fp.stem

        df = read_compiled_station(fp)
        hourly = hourly_aggregate(df)
        
        # ---- FORCE SHIFT (post-aggregation) ----
        if SHIFT_HOURS != 0 and len(hourly.index) > 0:
            hourly = hourly.copy()
            hourly.index = hourly.index + SHIFT
        # ---------------------------------------


        # Add time_utc column for convenience
        out_df = hourly.reset_index()
        out_df["time_utc"] = out_df["time_local"].dt.tz_convert("UTC").apply(to_iso_z)
        out_df["time_local"] = out_df["time_local"].dt.strftime("%Y-%m-%d %H:%M:%S%z")

        # Write station file
        out_fp = PER_STATION_DIR / f"{station}.hourly_mm.csv"
        out_df = out_df[["time_local", "time_utc", "rain_mm", "n_records", "remark"]]
        out_df.to_csv(out_fp, index=False)

        # Summary
        present = hourly["rain_mm"].notna()
        n_hours_present = int(present.sum())
        n_hours_total = int(len(hourly))
        start_local = hourly.index.min()
        end_local = hourly.index.max()

        summary_rows.append(
            {
                "station": station,
                "start_local": (start_local.strftime("%Y-%m-%d %H:%M:%S%z") if pd.notna(start_local) else ""),
                "end_local": (end_local.strftime("%Y-%m-%d %H:%M:%S%z") if pd.notna(end_local) else ""),
                "n_hours_total": n_hours_total,
                "n_hours_present": n_hours_present,
                "n_hours_missing": int((~present).sum()) if MISSING_AS_NAN else 0,
                "n_nonzero_hours": int((hourly["rain_mm"].fillna(0) > 0).sum()),
                "per_station_csv": str(out_fp),
            }
        )

    pd.DataFrame(summary_rows).to_csv(SUMMARY_CSV, index=False)

    print(f"Input:  {INPUT_DIR}")
    print(f"Output: {PER_STATION_DIR}")
    print(f"Summary: {SUMMARY_CSV}")
    print(f"Stations processed: {len(files)}")
    print(f"Hourly TZ: {LOCAL_TZ} (DST-aware)")
    print(f"Missing hours kept as NaN: {MISSING_AS_NAN}")


if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        print(f"ERROR: {e}", file=sys.stderr)
        sys.exit(1)


Input:  /mnt/12TB/Sujan/Spatial_correlation/Raw_rain_measurement/OneDrive_1_1-26-2026/stormwatch_compiled_rawstyle_mm
Output: /mnt/12TB/Sujan/Spatial_correlation/Codes/Compiled_rain/compiled_rawstyle_hourly_mm/per_station_hourly
Summary: /mnt/12TB/Sujan/Spatial_correlation/Codes/Compiled_rain/compiled_rawstyle_hourly_mm/hourly_coverage_summary.csv
Stations processed: 162
Hourly TZ: America/Chicago (DST-aware)
Missing hours kept as NaN: True
